# 🧱 Base analítica: do lake à tabela de modelagem

Este notebook monta a **tabela analítica** que alimentará os modelos: cada linha é um aluno avaliado, com a variável resposta (alfabetizado ou não) e o contexto do seu município. É a primeira etapa do projeto e a fundação de todas as seguintes, porque define a unidade de predição, as variáveis disponíveis e a estratégia que impede o vazamento de dados.

> **Convenção de trabalho:** o desenvolvimento acontece neste notebook, célula a célula, com os resultados salvos. Ao final da etapa, o código estável é promovido para `src/preprocessing/prod_01_base_analitica.py`.

**Decisões que governam este notebook** (ver [diário de decisões](../docs/decisoes.md)):
- **D-001, grão de modelagem:** a linha vem da camada Silver (o aluno), o contexto vem da camada Gold e das tabelas municipais (o território), sempre defasado no tempo;
- **D-002, dependência entre as fases:** este projeto verifica o contrato do lake construído na fase anterior, sem reexecutar aquela pipeline.

---

**Estratégia deste notebook:**

```
verificar o       ->  isolar a          ->  montar o          ->  integrar e   ->  desenhar o    ->  gravar a
lake (pré-voo)        população            contexto              auditar          split             base
Seção 1               Seção 2              defasado (3)          Seções 4 e 5     Seção 6           Seção 7
```

O princípio é o mesmo da fase anterior: cada passo é conferido contra um número esperado antes do seguinte, e nada é gravado sem verificação.

## 1. Setup e verificação de pré-requisitos

**Passos desta seção:** (1.1) configurar o acesso ao data lake da fase anterior; (1.2) verificar se as camadas exigidas existem e têm as colunas esperadas.

🎓 **Conceito, contrato entre pipelines:** a pipeline de dados da fase anterior é *upstream*; este projeto de machine learning é *downstream*. Quem está a jusante **verifica** se o insumo existe no formato esperado, e não reexecuta a pipeline de cima: reexecutar acoplaria os dois repositórios e duplicaria responsabilidade. Se algo faltar, a execução para com uma mensagem que orienta o que rodar antes (decisão D-002).

> 📌 **Nota para reprodução:** este notebook lê `config/config.json`, que não é versionado. Crie o seu a partir de `config/config.example.json`, apontando para o projeto e o bucket onde a pipeline da fase anterior foi executada.

In [47]:
# --- 1.1 Configuração e acesso ao data lake ---
import json
from pathlib import Path

import pandas as pd
import pydata_google_auth
from google.cloud import storage

CFG = json.loads(Path("../config/config.json").read_text(encoding="utf-8"))
PROJETO_GCP = CFG["projeto_gcp"]
BUCKET_LAKE = CFG["bucket_lake"]

ESCOPOS = ["https://www.googleapis.com/auth/cloud-platform"]
credenciais = pydata_google_auth.get_user_credentials(ESCOPOS)
credenciais = credenciais.with_quota_project(PROJETO_GCP)
cliente_storage = storage.Client(project=PROJETO_GCP, credentials=credenciais)

def garantir_credencial(forcar: bool = False) -> None:
    """Renova o token de acesso e limpa o cache do gcsfs.

    O token de usuário expira em cerca de uma hora. Em sessões longas de
    notebook, qualquer leitura ou gravação no lake feita depois disso
    falha com erro 401. Esta função deve ser chamada antes de cada acesso;
    o parâmetro forcar permite renovar mesmo quando o token ainda parece
    válido, situação em que o gcsfs pode manter em cache uma instância
    autenticada com o token anterior.
    """
    import google.auth.transport.requests
    import gcsfs
    if forcar or not credenciais.valid:
        credenciais.refresh(google.auth.transport.requests.Request())
        gcsfs.GCSFileSystem.clear_instance_cache()

def ultima_particao(area: str, tabela: str) -> str | None:
    """Partição mais recente de uma tabela do lake, ou None se não existir."""
    particoes = sorted({
        b.name.split("/")[2]
        for b in cliente_storage.list_blobs(BUCKET_LAKE, prefix=f"{area}/{tabela}/")
        if len(b.name.split("/")) > 2
    })
    return particoes[-1] if particoes else None

def ler_lake(area: str, tabela: str, **kwargs) -> pd.DataFrame:
    """Lê a partição mais recente de uma tabela do lake."""
    garantir_credencial()
    particao = ultima_particao(area, tabela)
    if particao is None:
        raise FileNotFoundError(
            f"Tabela '{tabela}' não encontrada em {area}/. "
            "Execute antes a pipeline da fase anterior."
        )
    caminho = f"gs://{BUCKET_LAKE}/{area}/{tabela}/{particao}/{tabela}.parquet"
    return pd.read_parquet(caminho, storage_options={"token": credenciais},
                           **kwargs)

print(f"Lake: gs://{BUCKET_LAKE}/")
print(f"Projeto GCP: {PROJETO_GCP}")
print("Setup ok")

Lake: gs://tech-challenge-fase2-lake-rm373453/
Projeto GCP: tech-challenge-fase2
Setup ok


In [48]:
# --- 1.2 Verificação de pré-requisitos (pré-voo) ---
# Contrato: tabelas e colunas que este projeto exige da fase anterior.
CONTRATO = {
    ("silver", "alunos"): ["ano", "id_municipio", "rede_nome", "presente",
                           "alfabetizado", "proficiencia", "peso_aluno"],
    ("silver", "municipio"): ["ano", "id_municipio", "rede", "nome",
                              "sigla_uf", "nome_regiao",
                              "taxa_alfabetizacao", "media_portugues"],
    ("silver", "metas_municipio"): ["ano_referencia", "id_municipio",
                                    "ano_meta", "meta_taxa"],
    ("gold", "indicador_municipio"): ["ano", "id_municipio", "taxa",
                                      "percentual_participacao",
                                      "meta_taxa", "origem"],
}

print("Verificação de pré-requisitos no lake:")
print()
pendencias = []
for (area, tabela), colunas in CONTRATO.items():
    particao = ultima_particao(area, tabela)
    if particao is None:
        print(f"  {area}/{tabela:<22} AUSENTE")
        pendencias.append(f"{area}/{tabela}")
        continue
    amostra = ler_lake(area, tabela, columns=colunas[:1])
    disponiveis = set(pd.read_parquet(
        f"gs://{BUCKET_LAKE}/{area}/{tabela}/{particao}/{tabela}.parquet",
        storage_options={"token": credenciais}).columns)
    faltantes = [c for c in colunas if c not in disponiveis]
    status = "OK" if not faltantes else f"COLUNAS FALTANTES: {faltantes}"
    if faltantes:
        pendencias.append(f"{area}/{tabela}: {faltantes}")
    print(f"  {area}/{tabela:<22} {particao}  {len(amostra):>9,} linhas  {status}")

print()
if pendencias:
    raise RuntimeError(
        "Pré-requisitos não atendidos: " + "; ".join(pendencias) + ".\n"
        "Execute a pipeline da fase anterior (repositório "
        "Tech_Challenge_RM373453_pipeline_alfabetizacao, passos 4 a 8 do "
        "Como Executar) antes de prosseguir."
    )
print("Contrato atendido: o lake tem o que este projeto precisa.")

Verificação de pré-requisitos no lake:

  silver/alunos                 data_processamento=2026-07-11  3,866,814 linhas  OK
  silver/municipio              data_processamento=2026-07-11     23,995 linhas  OK
  silver/metas_municipio        data_processamento=2026-07-11     74,928 linhas  OK
  gold/indicador_municipio    data_processamento=2026-07-12     11,629 linhas  OK

Contrato atendido: o lake tem o que este projeto precisa.


## 2. População de modelagem e variável resposta

**Passos desta seção:** (2.1) carregar os alunos da camada Silver e isolar a população modelável; (2.2) examinar a variável resposta e seu balanceamento.

🎓 **Conceito, população modelável** (decisão D-001): o modelo aprende com quem tem resultado observado. Alunos ausentes na avaliação não têm proficiência registrada e, portanto, não têm variável resposta: eles permanecem no lake, com a flag de presença criada na fase anterior, mas ficam fora do treinamento. Essa exclusão é uma premissa declarada, não um descarte silencioso, e volta como limitação do projeto: o modelo prevê a alfabetização de quem realiza a prova.

⚠️ **Achado do reconhecimento do lake:** o identificador de aluno se repete entre os ciclos de 2023 e 2024 em 86,7% dos casos, embora a avaliação seja aplicada a coortes diferentes (alunos do 2º ano de cada ano). O identificador é, portanto, uma máscara reutilizada, e não permite acompanhar a mesma criança ao longo do tempo. Isso exclui qualquer variável de trajetória individual e reforça que o identificador não entra no modelo.

In [49]:
# --- 2.1 Carregar alunos e isolar a população modelável ---
df_alunos = ler_lake(
    "silver", "alunos",
    columns=["ano", "id_municipio", "rede_nome", "presente",
             "alfabetizado", "proficiencia", "peso_aluno"],
)
print(f"Alunos na camada Silver: {len(df_alunos):,}")

# A população modelável: presentes na avaliação (D-001)
df_pop = df_alunos[df_alunos["presente"]].copy()
print(f"Presentes (com resultado observado): {len(df_pop):,} "
      f"({len(df_pop) / len(df_alunos):.1%})")
print(f"Ausentes (fora do treinamento):      "
      f"{len(df_alunos) - len(df_pop):,}")
print()

# Variável resposta: alfabetizado (1) ou não (0)
df_pop["alvo"] = (df_pop["alfabetizado"].astype(str) == "1").astype(int)

print("Distribuição por ciclo:")
print(df_pop.groupby("ano")["alvo"]
      .agg(alunos="size", alfabetizados="sum",
           taxa=lambda s: f"{100 * s.mean():.1f}%").to_string())

Alunos na camada Silver: 3,866,814
Presentes (com resultado observado): 3,354,661 (86.8%)
Ausentes (fora do treinamento):      512,153

Distribuição por ciclo:
       alunos  alfabetizados   taxa
ano                                
2023  1502809         877427  58.4%
2024  1851852        1107119  59.8%


In [50]:
# --- 2.2 Conferir a variável resposta contra o dado oficial ---
# A taxa da população modelável é a taxa NÃO ponderada; a oficial usa o
# peso amostral. As duas devem ficar próximas, e a diferença entre elas
# já antecipa a decisão pendente sobre o uso do peso no treinamento.

for ano in sorted(df_pop["ano"].unique()):
    recorte = df_pop[df_pop["ano"] == ano]
    simples = 100 * recorte["alvo"].mean()
    ponderada = (100 * (recorte["peso_aluno"] * recorte["alvo"]).sum()
                 / recorte["peso_aluno"].sum())
    print(f"{ano}:  simples {simples:.1f}%   ponderada {ponderada:.1f}%   "
          f"diferença {ponderada - simples:+.1f} pp")

print()
print("Referência oficial da rede pública (INEP): 55,9% em 2023 e 59,2% em 2024.")
print("A população acima inclui a rede privada, o que eleva ambas as taxas.")
print()
print("Distribuição por rede (todos os ciclos):")
print(df_pop.groupby("rede_nome")["alvo"]
      .agg(alunos="size", taxa=lambda s: f"{100 * s.mean():.1f}%").to_string())

2023:  simples 58.4%   ponderada 57.5%   diferença -0.9 pp
2024:  simples 59.8%   ponderada 59.2%   diferença -0.6 pp

Referência oficial da rede pública (INEP): 55,9% em 2023 e 59,2% em 2024.
A população acima inclui a rede privada, o que eleva ambas as taxas.

Distribuição por rede (todos os ciclos):
            alunos   taxa
rede_nome                
Estadual    372596  62.1%
Municipal  2982041  58.8%
Privada         24  66.7%


### 2.3 Dois achados que orientam as próximas decisões

- **O peso amostral reproduz o número oficial.** A taxa ponderada de 2024 (59,2%) coincide com a taxa divulgada pelo INEP para a rede pública naquele ciclo, enquanto a taxa simples fica 0,6 ponto acima. O peso, calibrado pelo instituto, corrige a sub-representação de certos perfis na amostra: é a evidência que sustentará a decisão sobre usá-lo como ponderação no treinamento.
- **A rede privada é residual nesta base:** 24 alunos entre 3,35 milhões. Não constitui uma categoria com massa estatística; será tratada de forma explícita na preparação, para não gerar uma classe rara sem significado no modelo.

## 3. Contexto defasado: da rede do aluno e do seu município

**Passos desta seção:** (3.1) montar o retrato da rede de ensino de cada município no ciclo anterior; (3.2) acrescentar o retrato do município como um todo e as variáveis conhecidas antes da avaliação.

🎓 **Conceito, o que o modelo pode saber** (*informação ex-ante × ex-post*): a regra que separa uma variável legítima de um vazamento não é o assunto dela, é o **momento em que ela passa a existir**. No instante da predição já se conhece a situação do território no ciclo anterior e a meta pactuada para o ciclo corrente; ainda não se conhece o resultado do próprio ciclo.

| Variável | Quando passa a existir | Entra no modelo |
|---|---|---|
| Desempenho da rede e do município no ciclo **anterior** | antes da avaliação | ✅ sim |
| **Meta** pactuada para o ciclo corrente | antes da avaliação (é um pacto prévio) | ✅ sim |
| Taxa do município no ciclo **corrente** | depois da avaliação, e calculada **com o próprio aluno** | ❌ não |

O caso da meta merece destaque: ela se refere ao ciclo que se quer prever, mas é **conhecida de antemão**, porque foi pactuada entre os entes federativos antes da aplicação da prova. Não é vazamento; é exatamente a informação que um gestor teria em mãos ao tentar antecipar o resultado.

🎓 **Conceito, o grão do contexto:** o aluno pertence a uma **rede** dentro de um **município**, e as duas camadas informam coisas diferentes. A rede estadual e a rede municipal de uma mesma cidade divergem mais do que se imagina: nos 1.083 municípios em que ambas foram medidas em 2023, a diferença entre elas tem desvio padrão de 19,5 pontos percentuais e supera 10 pontos em 58% dos casos. Usar apenas o agregado das duas descartaria essa variação e trataria alunos de realidades distintas como se vivessem a mesma. Por isso o contexto é montado em dois níveis:

- **rede do aluno**, o desempenho da rede que efetivamente o atende;
- **município**, o clima educacional do território como um todo, com participação e porte;
- e a **diferença entre os dois**, que posiciona a rede do aluno acima ou abaixo do seu município.

📌 **Consequência do desenho temporal:** como a camada Silver cobre 2023 e 2024, apenas o ciclo de **2024** reúne aluno e contexto anterior. O ciclo de 2023 **não entra como linha, e sim como contexto**: o retrato daquele ano vira coluna em todas as observações. Isso evita dois erros opostos, o de misturar ciclos deixando metade das linhas sem contexto, e o de usar o contexto do próprio ciclo, que seria vazamento.

In [51]:
# --- 3.1 Retrato da rede de ensino no ciclo anterior ---
CICLO_ALVO = 2024
CICLO_ANTERIOR = CICLO_ALVO - 1

mun = ler_lake("silver", "municipio",
               columns=["ano", "id_municipio", "rede", "rede_nome",
                        "taxa_alfabetizacao", "media_portugues",
                        "sigla_uf", "nome_regiao"])

# Contexto no grão da rede: a rede que atende o aluno (Estadual ou Municipal)
ctx_rede = mun.loc[
    (mun["ano"] == CICLO_ANTERIOR) & (mun["rede"].astype(str).isin(["2", "3"])),
    ["id_municipio", "rede_nome", "taxa_alfabetizacao", "media_portugues"],
].rename(columns={
    "taxa_alfabetizacao": "rede_taxa_ant",
    "media_portugues": "rede_media_portugues_ant",
})

print(f"Retratos de rede em {CICLO_ANTERIOR}: {len(ctx_rede):,}")
print(ctx_rede.groupby("rede_nome", observed=True)
      .agg(municipios=("id_municipio", "nunique"),
           taxa_media=("rede_taxa_ant", lambda s: f"{s.mean():.1f}%"))
      .to_string())
print()

# Quanto as redes divergem dentro do mesmo município
comparativo = ctx_rede.pivot_table(index="id_municipio", columns="rede_nome",
                                   values="rede_taxa_ant", observed=True)
ambas = comparativo.dropna()
diferenca = ambas["Estadual"] - ambas["Municipal"]
print(f"Municípios com as duas redes medidas: {len(ambas):,}")
print(f"Diferença Estadual - Municipal: média {diferenca.mean():+.1f} pp, "
      f"desvio {diferenca.std():.1f} pp")
print(f"Acima de 10 pp de diferença: {100 * (diferenca.abs() > 10).mean():.0f}% "
      f"dos municípios")

Retratos de rede em 2023: 6,597
           municipios taxa_media
rede_nome                       
Estadual         1149      63.8%
Municipal        5448      60.3%

Municípios com as duas redes medidas: 1,083
Diferença Estadual - Municipal: média +3.7 pp, desvio 19.5 pp
Acima de 10 pp de diferença: 58% dos municípios


In [52]:
# --- 3.2 Retrato do município e variáveis conhecidas antes da avaliação ---
# Município como um todo (rede pública), vindo da camada Gold
gold = ler_lake("gold", "indicador_municipio")
ctx_mun = gold.loc[
    (gold["ano"] == CICLO_ANTERIOR) & (gold["origem"] == "oficial_inep"),
    ["id_municipio", "taxa", "percentual_participacao", "taxa_ajustada",
     "alunos_presentes"],
].rename(columns={
    "taxa": "mun_taxa_ant",
    "percentual_participacao": "mun_participacao_ant",
    "taxa_ajustada": "mun_taxa_ajustada_ant",
    "alunos_presentes": "mun_alunos_ant",
})

# Território: unidade da federação e região (estáveis no tempo)
territorio = mun.loc[
    (mun["ano"] == CICLO_ANTERIOR) & (mun["rede"].astype(str) == "5"),
    ["id_municipio", "sigla_uf", "nome_regiao"],
]
ctx_mun = ctx_mun.merge(territorio, on="id_municipio", how="left")

# Meta pactuada para o ciclo alvo: informação ex-ante
metas = gold.loc[
    (gold["ano"] == CICLO_ALVO) & (gold["origem"] == "oficial_inep"),
    ["id_municipio", "meta_taxa"],
].rename(columns={"meta_taxa": "mun_meta_ciclo"})
ctx_mun = ctx_mun.merge(metas, on="id_municipio", how="left")
ctx_mun["mun_gap_meta"] = ctx_mun["mun_meta_ciclo"] - ctx_mun["mun_taxa_ant"]

# Espaço reservado para o enriquecimento externo (etapa 3 do plano):
# fontes municipais novas entram aqui, por join em id_municipio.

print(f"Contexto municipal: {len(ctx_mun):,} municípios, "
      f"{len(ctx_mun.columns) - 1} variáveis")
print()
print("Preenchimento das variáveis municipais:")
print(pd.DataFrame({
    "% preenchido": (100 * ctx_mun.notna().mean()).round(1)
}).drop(index="id_municipio").to_string())

Contexto municipal: 4,950 municípios, 8 variáveis

Preenchimento das variáveis municipais:
                       % preenchido
mun_taxa_ant                  100.0
mun_participacao_ant           98.4
mun_taxa_ajustada_ant          98.4
mun_alunos_ant                 98.4
sigla_uf                      100.0
nome_regiao                   100.0
mun_meta_ciclo                 94.7
mun_gap_meta                   94.7


### 3.3 Benchmark estadual: comparar a rede com os seus pares

🎓 **Conceito, posição relativa ao grupo de pares:** o valor absoluto de um indicador diz pouco sem referência. Uma rede municipal com 60% de alfabetização representa uma situação boa em um estado cuja mediana é 50%, e ruim em outro cuja mediana é 70%. O que informa é a **posição relativa**, e os pares corretos para comparação são as redes do **mesmo tipo**, no **mesmo estado**, porque compartilham política estadual, contexto socioeconômico e regime de colaboração entre os entes.

Duas variáveis nascem daí: o benchmark em si (a mediana da rede na unidade da federação) e a distância da rede do aluno até ele. A mediana é preferida à média por ser robusta a municípios atípicos, comuns em estados com poucas redes medidas.

📌 **Sem vazamento:** o benchmark é calculado sobre o ciclo anterior, o mesmo do restante do contexto. Ele não contém informação do ciclo que se quer prever.

In [53]:
# --- 3.3 Benchmark estadual por rede ---
# Mediana da taxa de cada rede entre os municípios da mesma UF (ciclo anterior)
base_bench = ctx_rede.merge(
    mun.loc[mun["ano"] == CICLO_ANTERIOR, ["id_municipio", "sigla_uf"]]
       .drop_duplicates(),
    on="id_municipio", how="left")

benchmark = (base_bench.groupby(["sigla_uf", "rede_nome"], observed=True)
             ["rede_taxa_ant"].median()
             .rename("uf_rede_taxa_ant").reset_index())

print(f"Benchmarks calculados: {len(benchmark)} combinações de UF e rede")
print()
print("Amostra (as cinco maiores e as cinco menores medianas):")
ordenado = benchmark.sort_values("uf_rede_taxa_ant", ascending=False)
print(pd.concat([ordenado.head(5), ordenado.tail(5)])
      .round(1).to_string(index=False))
print()

# Acoplar o benchmark ao contexto de rede
ctx_rede = base_bench.merge(benchmark, on=["sigla_uf", "rede_nome"], how="left")
ctx_rede["rede_vs_uf"] = (ctx_rede["rede_taxa_ant"]
                          - ctx_rede["uf_rede_taxa_ant"])
ctx_rede = ctx_rede.drop(columns="sigla_uf")

print("Distância da rede até o benchmark do seu estado (pontos percentuais):")
print(ctx_rede["rede_vs_uf"].describe().round(1).to_string())

Benchmarks calculados: 46 combinações de UF e rede

Amostra (as cinco maiores e as cinco menores medianas):
sigla_uf rede_nome  uf_rede_taxa_ant
      CE Municipal              93.2
      PR  Estadual              84.7
      CE  Estadual              84.2
      GO  Estadual              79.5
      ES  Estadual              79.1
      BA Municipal              36.1
      RN Municipal              36.0
      AL  Estadual              30.1
      SE Municipal              28.9
      BA  Estadual              27.5

Distância da rede até o benchmark do seu estado (pontos percentuais):
count    6597.0
mean        0.3
std        15.4
min       -66.4
25%        -9.6
50%         0.0
75%         9.7
max        58.2


### 3.4 Porte do ciclo corrente: o que já se sabe antes da prova

🎓 **Conceito, nem tudo do ciclo corrente é vazamento.** A regra continua sendo o momento em que a informação passa a existir. O **porte** da rede e do município no ciclo que se quer prever, isto é, quantos alunos há para avaliar, vem do cadastro escolar e está definido **antes** da aplicação da prova. Não depende de nenhum resultado, e por isso é informação legítima.

⚠️ **A linha fina:** alunos **avaliáveis** (matriculados, presentes e ausentes) é informação prévia; alunos **presentes** só se conhece depois da aplicação, e seria vazamento. O porte aqui conta os avaliáveis.

📌 **Por que preferir o porte corrente ao do ciclo anterior:** o porte defasado sofre de dois problemas. Envelhece (redes crescem e encolhem entre ciclos) e depende da disponibilidade dos microdados do ano anterior, o que restringe sua cobertura a 76,9% das observações. O porte corrente é calculado da própria base de alunos e cobre a totalidade. Não à toa, o porte defasado foi a variável com a menor correlação com a resposta entre todas as testadas.

Da mesma estrutura nasce uma terceira variável, que nenhuma outra expressa: a **fração do município atendida pela rede do aluno**. Ela distingue quem estuda na rede predominante do território de quem está em uma rede minoritária ali.

In [54]:
# --- 3.4 Porte do ciclo corrente (informação prévia à avaliação) ---
# Base completa do ciclo (presentes e ausentes): é o cadastro de avaliáveis
cadastro = df_alunos[df_alunos["ano"] == CICLO_ALVO]

porte_rede = (cadastro.groupby(["id_municipio", "rede_nome"], observed=True)
              .size().rename("rede_porte_atual").reset_index())
porte_mun = (cadastro.groupby("id_municipio", observed=True)
             .size().rename("mun_porte_atual").reset_index())

print(f"Avaliáveis no ciclo {CICLO_ALVO}: {len(cadastro):,} "
      f"(presentes e ausentes)")
print(f"Combinações município e rede: {len(porte_rede):,}")
print(f"Municípios: {len(porte_mun):,}")
print()
print("Porte por rede (alunos avaliáveis por município):")
print(porte_rede.groupby("rede_nome", observed=True)["rede_porte_atual"]
      .describe()[["count", "mean", "50%", "max"]].round(0).to_string())

Avaliáveis no ciclo 2024: 2,119,624 (presentes e ausentes)
Combinações município e rede: 6,543
Municípios: 5,519

Porte por rede (alunos avaliáveis por município):
            count   mean    50%      max
rede_nome                               
Estadual   1090.0  257.0   49.0  58612.0
Municipal  5452.0  337.0  114.0  49820.0
Privada       1.0   25.0   25.0     25.0


## 4. Integração e auditoria contra vazamento

**Passos desta seção:** (4.1) integrar cada aluno ao contexto da sua rede e do seu município, conferindo a cobertura; (4.2) submeter a tabela a uma auditoria explícita contra vazamento.

🎓 **Conceito, vazamento de dados** (*data leakage*): ocorre quando uma variável explicativa carrega informação que, no momento real da predição, ainda não existiria. O modelo aprende um atalho, exibe métricas excelentes na avaliação e falha no uso real. O enunciado exige o tratamento desse problema, e aqui ele tem **três fontes distintas**, cada uma com sua defesa:

| Fonte de vazamento | Como se manifestaria aqui | Defesa adotada |
|---|---|---|
| **Temporal** | usar o desempenho do território no próprio ciclo do aluno | contexto sempre defasado (seção 3) |
| **Da variável resposta** | usar a proficiência do aluno, da qual a resposta é derivada | exclusão explícita, auditada na célula 4.2 |
| **Entre partições** | alunos do mesmo município em treino e em teste | separação por município (seção 5) |

⚠️ **A armadilha mais perigosa é a segunda.** A variável resposta é derivada da proficiência (alfabetizado quando a nota atinge 743 pontos). Se a proficiência entrar como variável explicativa, o modelo atinge acurácia praticamente perfeita sem aprender nada: é o gabarito dentro da prova. Por isso a auditoria mantém uma lista de colunas proibidas e verifica, em código, que nenhuma delas alcançou a tabela final.

In [55]:
# --- 4.1 Integrar aluno, rede e município ---
alunos_ciclo = df_pop[df_pop["ano"] == CICLO_ALVO].copy()
print(f"Alunos presentes em {CICLO_ALVO}: {len(alunos_ciclo):,}")

# Primeiro nível: o contexto da rede que atende o aluno
antes = len(alunos_ciclo)
abt = alunos_ciclo.merge(ctx_rede, on=["id_municipio", "rede_nome"], how="left")
assert len(abt) == antes, "o join com a rede alterou a contagem de linhas"

# Segundo nível: o contexto do município como um todo
abt = abt.merge(ctx_mun, on="id_municipio", how="left")
assert len(abt) == antes, "o join com o município alterou a contagem de linhas"
print(f"Linhas após os dois joins: {len(abt):,}  (esperado: igual)")
print()

# Terceiro nível: o porte do ciclo corrente (informação prévia)
abt = abt.merge(porte_rede, on=["id_municipio", "rede_nome"], how="left")
abt = abt.merge(porte_mun, on="id_municipio", how="left")
assert len(abt) == antes, "o join com o porte alterou a contagem de linhas"

# Derivadas: posição da rede no território e peso dela no município
abt["rede_vs_municipio"] = abt["rede_taxa_ant"] - abt["mun_taxa_ant"]
abt["rede_peso_no_municipio"] = (abt["rede_porte_atual"]
                                 / abt["mun_porte_atual"])

# Cobertura de cada nível de contexto
print("Cobertura do contexto:")
for rotulo, coluna in [("da rede do aluno", "rede_taxa_ant"),
                       ("do município", "mun_taxa_ant")]:
    cob = abt[coluna].notna()
    print(f"  {rotulo:<18} {cob.sum():>9,} alunos  ({cob.mean():.1%})")
print()
print("Cobertura da rede, por rede de ensino:")
print(abt.groupby("rede_nome", observed=True)["rede_taxa_ant"]
      .agg(alunos="size",
           com_contexto=lambda s: f"{100 * s.notna().mean():.1f}%").to_string())
print()
print("Preenchimento das variáveis na tabela analítica:")
print(pd.DataFrame({"% preenchido": (100 * abt.notna().mean()).round(1)})
      .sort_values("% preenchido").to_string())

Alunos presentes em 2024: 1,851,852
Linhas após os dois joins: 1,851,852  (esperado: igual)

Cobertura do contexto:
  da rede do aluno   1,816,270 alunos  (98.1%)
  do município       1,652,243 alunos  (89.2%)

Cobertura da rede, por rede de ensino:
            alunos com_contexto
rede_nome                      
Estadual    241074        88.7%
Municipal  1610754        99.5%
Privada         24         0.0%

Preenchimento das variáveis na tabela analítica:
                          % preenchido
mun_alunos_ant                    76.9
mun_participacao_ant              76.9
mun_taxa_ajustada_ant             76.9
mun_gap_meta                      86.5
mun_meta_ciclo                    86.5
nome_regiao                       89.2
sigla_uf                          89.2
mun_taxa_ant                      89.2
rede_vs_municipio                 89.2
rede_taxa_ant                     98.1
uf_rede_taxa_ant                  98.1
rede_media_portugues_ant          98.1
rede_vs_uf                       

In [56]:
# --- 4.2 Auditoria contra vazamento ---
PROIBIDAS = {
    "proficiencia": "a variável resposta é derivada dela (nota >= 743)",
    "alfabetizado": "é a própria variável resposta, em outro formato",
    "presente": "constante na população modelada (todos são presentes)",
    "peso_aluno": "metadado amostral; reservado à ponderação, não é atributo",
}

CHAVES = ["ano", "id_municipio"]
FEATURES_ALUNO = ["rede_nome"]
FEATURES_REDE = ["rede_taxa_ant", "rede_media_portugues_ant",
                 "uf_rede_taxa_ant", "rede_vs_uf", "rede_vs_municipio"]
FEATURES_MUNICIPIO = [c for c in ctx_mun.columns if c != "id_municipio"]
FEATURES_PORTE = ["rede_porte_atual", "mun_porte_atual",
                  "rede_peso_no_municipio"]
FEATURES = (FEATURES_ALUNO + FEATURES_REDE + FEATURES_MUNICIPIO
            + FEATURES_PORTE)
RESPOSTA = "alvo"

print("Auditoria contra vazamento")
print("=" * 60)

invasoras = [c for c in FEATURES if c in PROIBIDAS]
print(f"1. Colunas proibidas entre as variáveis: {invasoras or 'nenhuma'}")
for coluna, motivo in PROIBIDAS.items():
    print(f"     {coluna:<14} fora do modelo, {motivo}")

# Variáveis do ciclo corrente admitidas, por serem prévias à avaliação:
# a meta pactuada e o porte do cadastro escolar. Qualquer outra é suspeita.
EX_ANTE_DO_CICLO = set(FEATURES_PORTE) | {"mun_meta_ciclo", "mun_gap_meta"}
DERIVADAS = {"rede_vs_municipio", "rede_vs_uf"}
suspeitas = [c for c in FEATURES
             if not c.endswith("_ant") and c not in EX_ANTE_DO_CICLO
             and c not in DERIVADAS
             and c not in ("rede_nome", "sigla_uf", "nome_regiao")]
print(f"2. Variáveis do ciclo corrente sem justificativa prévia: "
      f"{suspeitas or 'nenhuma'}")
print("     admitidas por serem anteriores à prova: meta pactuada e "
      "porte do cadastro")

numericas = [c for c in FEATURES if pd.api.types.is_numeric_dtype(abt[c])]
correl = abt[numericas + [RESPOSTA]].corr()[RESPOSTA].drop(RESPOSTA)
print("3. Correlação com a variável resposta "
      "(valores extremos indicariam vazamento):")
print(correl.sort_values(key=abs, ascending=False).round(3).to_string())
suspeita_alta = correl[correl.abs() > 0.9]
print(f"   correlações acima de 0,9: {list(suspeita_alta.index) or 'nenhuma'}")

print("=" * 60)
if invasoras or suspeitas or len(suspeita_alta):
    raise RuntimeError("Auditoria reprovada: revise as variáveis acima.")
print(f"Auditoria aprovada. {len(FEATURES)} variáveis explicativas, "
      f"{len(abt):,} observações.")

Auditoria contra vazamento
1. Colunas proibidas entre as variáveis: nenhuma
     proficiencia   fora do modelo, a variável resposta é derivada dela (nota >= 743)
     alfabetizado   fora do modelo, é a própria variável resposta, em outro formato
     presente       fora do modelo, constante na população modelada (todos são presentes)
     peso_aluno     fora do modelo, metadado amostral; reservado à ponderação, não é atributo
2. Variáveis do ciclo corrente sem justificativa prévia: nenhuma
     admitidas por serem anteriores à prova: meta pactuada e porte do cadastro
3. Correlação com a variável resposta (valores extremos indicariam vazamento):
mun_taxa_ajustada_ant       0.275
mun_taxa_ant                0.248
rede_media_portugues_ant    0.247
rede_taxa_ant               0.246
mun_meta_ciclo              0.232
uf_rede_taxa_ant            0.192
mun_gap_meta               -0.189
mun_participacao_ant        0.151
rede_vs_uf                  0.108
rede_vs_municipio           0.057
rede_pe

### 4.3 O que a integração revelou

O contexto no grão da rede cobre mais alunos do que o agregado municipal cobria, porque a maior parte da população estuda na rede municipal, que tem presença em praticamente todos os municípios avaliados. As lacunas remanescentes concentram-se em dois grupos: alunos da rede estadual em municípios cuja rede estadual não foi medida no ciclo anterior, e alunos da rede privada, residual nesta base e sem contexto correspondente.

As variáveis derivadas dos microdados no nível do município (participação, taxa ajustada e total de alunos presentes) permanecem com preenchimento menor, por causa dos municípios que constam no indicador consolidado sem microdados públicos, um ponto cego identificado na fase anterior. Como vários deles são populosos, o efeito sobre a contagem de alunos é maior do que sobre a de municípios.

O tratamento desses ausentes é assunto da etapa de pré-processamento, com duas alternativas a comparar: imputação por medida de tendência central dentro do próprio estado, ou descarte das variáveis que a análise exploratória mostrar pouco informativas. A decisão será registrada no diário com a evidência que a sustentar.

## 5. Partição dos dados: por município, não por sorteio

**Passos desta seção:** (5.1) separar treino, validação e teste por município; (5.2) conferir que a partição é honesta e equilibrada.

🎓 **Conceito, por que não o `train_test_split`:** a função mais conhecida do scikit-learn sorteia **linhas** ao acaso, e é a escolha correta quando cada observação é independente das demais. Não é o caso aqui: os alunos estão **aninhados em municípios**, e nove das dez variáveis explicativas são municipais, idênticas para todos os alunos de um mesmo território.

O efeito fica claro com um exemplo desta base. São Paulo contribui com cerca de cem mil alunos, todos com os mesmos valores de contexto. Num sorteio por linha, parte deles ficaria no treino e parte no teste; o modelo aprenderia a associação entre aquele conjunto específico de valores e o resultado, e voltaria a encontrá-lo na avaliação. A métrica subiria sem que houvesse generalização: o modelo teria memorizado o município, não aprendido o fenômeno. O efeito tem nome, **vazamento por agrupamento**, e é a terceira fonte listada na seção 4.

A resposta do próprio scikit-learn para dados agrupados é o `GroupShuffleSplit`, do mesmo módulo `model_selection`: em vez de sortear linhas, ele sorteia **grupos**, mantendo todas as observações de um município do mesmo lado da fronteira. O critério de escolha entre as duas funções é a estrutura do dado, não o costume:

| Estrutura das observações | Função adequada |
|---|---|
| Independentes (uma linha, uma entidade) | `train_test_split` |
| Aninhadas em grupos (alunos em municípios, consultas em pacientes) | `GroupShuffleSplit`, `GroupKFold` |

A partição por município resolve isso e, mais do que resolver, **mede o que interessa**: o conjunto de teste passa a ser formado por municípios que o modelo nunca viu, que é a situação real de uso, quando se pretende antecipar o risco de territórios ainda não avaliados no ciclo.

📌 **Proporção adotada:** 60% dos municípios para treino, 20% para validação e 20% para teste. A validação serve à escolha de modelos e ao ajuste de hiperparâmetros; o teste permanece intocado até a avaliação final, para preservar a honestidade da estimativa de desempenho.

In [57]:
# --- 5.1 Separar treino, validação e teste por município ---
from sklearn.model_selection import GroupShuffleSplit

SEMENTE = 42  # replicabilidade: mesma semente, mesma partição

municipios = abt["id_municipio"]

# Primeira divisão: 60% treino, 40% para dividir entre validação e teste
divisor = GroupShuffleSplit(n_splits=1, train_size=0.6, random_state=SEMENTE)
idx_treino, idx_resto = next(divisor.split(abt, groups=municipios))

# Segunda divisão: metade do resto para validação, metade para teste
resto = abt.iloc[idx_resto]
divisor2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEMENTE)
idx_val, idx_teste = next(divisor2.split(resto, groups=resto["id_municipio"]))

abt["particao"] = "treino"
abt.iloc[idx_resto, abt.columns.get_loc("particao")] = "teste"
abt.iloc[idx_resto[idx_val], abt.columns.get_loc("particao")] = "validacao"

print("Distribuição das partições:")
print(abt.groupby("particao")
      .agg(alunos=("alvo", "size"),
           municipios=("id_municipio", "nunique"),
           taxa_alfabetizacao=("alvo", lambda s: f"{100 * s.mean():.1f}%"))
      .to_string())

Distribuição das partições:
            alunos  municipios taxa_alfabetizacao
particao                                         
teste       311794        1104              60.8%
treino     1156013        3310              59.4%
validacao   384045        1103              60.0%


In [58]:
# --- 5.2 Conferir que a partição é honesta ---
print("Verificações da partição")
print("=" * 58)

# 1. Nenhum município aparece em mais de uma partição
por_municipio = abt.groupby("id_municipio")["particao"].nunique()
vazados = (por_municipio > 1).sum()
print(f"1. Municípios em mais de uma partição: {vazados}  (esperado: 0)")

# 2. Todas as observações foram atribuídas
sem_particao = abt["particao"].isna().sum()
print(f"2. Observações sem partição: {sem_particao}  (esperado: 0)")

# 3. A variável resposta tem distribuição semelhante entre as partições
taxas = abt.groupby("particao")["alvo"].mean() * 100
amplitude = taxas.max() - taxas.min()
print(f"3. Taxa de alfabetização por partição: "
      f"{', '.join(f'{p} {t:.1f}%' for p, t in taxas.items())}")
print(f"   amplitude entre partições: {amplitude:.1f} pp "
      f"(diferenças pequenas são esperadas: a partição é por município)")

# 4. A cobertura de contexto é semelhante entre as partições
cobertura = abt.groupby("particao")["mun_taxa_ant"].apply(
    lambda s: 100 * s.notna().mean())
print("4. Alunos com contexto por partição: "
      f"{', '.join(f'{p} {c:.1f}%' for p, c in cobertura.items())}")

print("=" * 58)
if vazados or sem_particao:
    raise RuntimeError("Partição reprovada nas verificações acima.")
print("Partição aprovada: nenhum município atravessa as fronteiras.")

Verificações da partição
1. Municípios em mais de uma partição: 0  (esperado: 0)
2. Observações sem partição: 0  (esperado: 0)
3. Taxa de alfabetização por partição: teste 60.8%, treino 59.4%, validacao 60.0%
   amplitude entre partições: 1.4 pp (diferenças pequenas são esperadas: a partição é por município)
4. Alunos com contexto por partição: teste 90.7%, treino 90.2%, validacao 85.2%
Partição aprovada: nenhum município atravessa as fronteiras.


## 6. Gravação da tabela analítica

**Passos desta seção:** (6.1) selecionar as colunas finais e gravar a tabela no data lake; (6.2) reconciliar a gravação e registrar o dicionário de variáveis.

🎓 **Conceito, onde vive a tabela analítica:** o repositório versiona código, e os dados vivem no lake, princípio herdado da fase anterior. A tabela analítica é gravada em uma área própria (`ml/`), separada das camadas do medalhão, porque não é um produto de dados de negócio, e sim um artefato de modelagem, com público e ciclo de vida próprios. As etapas seguintes leem essa tabela, e não voltam às camadas originais.

📌 **O que a tabela carrega, e por quê:**

| Grupo | Colunas | Papel |
|---|---|---|
| Chaves | `ano`, `id_municipio` | rastreabilidade e agregação dos resultados |
| Explicativas | `rede_nome` e as nove variáveis municipais | entram no modelo |
| Resposta | `alvo` | o que se quer prever |
| Ponderação | `peso_aluno` | disponível para o treinamento ponderado, não é atributo |
| Partição | `particao` | preserva a separação entre treino, validação e teste em todas as etapas |

In [59]:
# --- 6.1 Selecionar as colunas finais e gravar no lake ---
from datetime import datetime, timezone

COLUNAS_ABT = CHAVES + FEATURES + [RESPOSTA, "peso_aluno", "particao"]
df_abt = abt[COLUNAS_ABT].copy()

print(f"Tabela analítica: {len(df_abt):,} linhas x {len(df_abt.columns)} colunas")
print(f"  chaves:      {CHAVES}")
print(f"  explicativas: {FEATURES}")
print(f"  resposta:    {RESPOSTA}   ponderação: peso_aluno   partição: particao")
print()

garantir_credencial()
momento = datetime.now(timezone.utc)
df_abt["_processing_timestamp"] = momento.isoformat()
destino = (f"gs://{BUCKET_LAKE}/ml/abt_alfabetizacao/"
           f"data_processamento={momento:%Y-%m-%d}/abt_alfabetizacao.parquet")
df_abt.to_parquet(destino, index=False,
                  storage_options={"token": credenciais})
print(f"Gravado em: {destino}")

Tabela analítica: 1,851,852 linhas x 22 colunas
  chaves:      ['ano', 'id_municipio']
  explicativas: ['rede_nome', 'rede_taxa_ant', 'rede_media_portugues_ant', 'uf_rede_taxa_ant', 'rede_vs_uf', 'rede_vs_municipio', 'mun_taxa_ant', 'mun_participacao_ant', 'mun_taxa_ajustada_ant', 'mun_alunos_ant', 'sigla_uf', 'nome_regiao', 'mun_meta_ciclo', 'mun_gap_meta', 'rede_porte_atual', 'mun_porte_atual', 'rede_peso_no_municipio']
  resposta:    alvo   ponderação: peso_aluno   partição: particao

Gravado em: gs://tech-challenge-fase2-lake-rm373453/ml/abt_alfabetizacao/data_processamento=2026-09-06/abt_alfabetizacao.parquet


In [60]:
# --- 6.2 Reconciliar a gravação e registrar o dicionário de variáveis ---
garantir_credencial()  # sessões longas: o token expira em cerca de uma hora
relido = pd.read_parquet(destino, storage_options={"token": credenciais})
status = "OK" if len(relido) == len(df_abt) else "DIVERGIU"
print(f"Reconciliação: gravado {len(df_abt):,} | relido {len(relido):,}  {status}")
print()

DESCRICOES = {
    "ano": "ciclo da avaliação (chave)",
    "id_municipio": "código IBGE do município (chave)",
    "rede_nome": "rede de ensino que atende o aluno",
    "rede_taxa_ant": "taxa de alfabetização da rede do aluno, no município, no ciclo anterior",
    "rede_media_portugues_ant": "proficiência média em português da rede do aluno no ciclo anterior",
    "uf_rede_taxa_ant": "mediana da taxa da mesma rede entre os municípios da UF (benchmark)",
    "rede_vs_uf": "distância entre a taxa da rede do aluno e o benchmark estadual da rede",
    "rede_vs_municipio": "diferença entre a taxa da rede do aluno e a do município no ciclo anterior",
    "mun_taxa_ant": "taxa de alfabetização do município (rede pública) no ciclo anterior",
    "mun_participacao_ant": "percentual de participação na avaliação anterior",
    "mun_taxa_ajustada_ant": "taxa anterior com ausentes contados como não alfabetizados",
    "mun_alunos_ant": "alunos presentes no município no ciclo anterior (porte defasado)",
    "sigla_uf": "unidade da federação",
    "nome_regiao": "região do país",
    "mun_meta_ciclo": "meta pactuada para o ciclo corrente (informação prévia)",
    "mun_gap_meta": "distância entre a meta do ciclo e a taxa do ciclo anterior",
    "rede_porte_atual": "alunos avaliáveis na rede do aluno, no município, no ciclo corrente",
    "mun_porte_atual": "alunos avaliáveis no município no ciclo corrente",
    "rede_peso_no_municipio": "fração dos alunos do município atendida pela rede do aluno",
    "alvo": "variável resposta: 1 se alfabetizado, 0 caso contrário",
    "peso_aluno": "peso amostral do INEP (ponderação, não é atributo)",
    "particao": "conjunto de destino: treino, validacao ou teste",
}

# Relações determinísticas entre variáveis: quem é derivável de quem.
# Documentá-las aqui é o que permite, na etapa seguinte, decidir com
# critério quais manter, evitando redundância e inflação de dimensão.
RELACOES = {
    "rede_vs_municipio": "= rede_taxa_ant - mun_taxa_ant",
    "rede_vs_uf": "= rede_taxa_ant - uf_rede_taxa_ant",
    "mun_gap_meta": "= mun_meta_ciclo - mun_taxa_ant",
    "rede_peso_no_municipio": "= rede_porte_atual / mun_porte_atual",
    "mun_taxa_ajustada_ant": "= mun_taxa_ant * mun_participacao_ant / 100",
}

variaveis = df_abt.columns.drop("_processing_timestamp")

# Trava: nenhuma variável da tabela pode ficar sem descrição
sem_descricao = [v for v in variaveis if v not in DESCRICOES]
orfas = [v for v in DESCRICOES if v not in variaveis]
if sem_descricao or orfas:
    raise RuntimeError(
        f"Dicionário desatualizado. Sem descrição: {sem_descricao}. "
        f"Descrições sem variável correspondente: {orfas}. "
        "Reexecute as seções 3 e 4 antes desta célula."
    )

dicionario = pd.DataFrame({"variavel": variaveis})
dicionario["descricao"] = dicionario["variavel"].map(DESCRICOES)
dicionario["derivada_de"] = dicionario["variavel"].map(RELACOES).fillna("")
dicionario["tipo"] = [str(df_abt[v].dtype) for v in dicionario["variavel"]]
dicionario["% preenchido"] = [
    round(100 * df_abt[v].notna().mean(), 1) for v in dicionario["variavel"]]

print(f"Dicionário de variáveis ({len(dicionario)} entradas, "
      "todas descritas):")
print(dicionario.to_string(index=False))

derivadas = dicionario[dicionario["derivada_de"] != ""]
print()
print(f"⚠️ {len(derivadas)} variáveis são funções determinísticas de outras. "
      "Elas permanecem na tabela")
print("como candidatas, e a análise exploratória decidirá quais entram no "
      "modelo, medindo")
print("correlação e inflação de variância. Diferenças e razões costumam "
      "ajudar modelos de")
print("árvore, que não as constroem sozinhos, e prejudicar modelos lineares, "
      "por colinearidade.")

dicionario.to_csv("../reports/dicionario_abt.csv", index=False,
                  encoding="utf-8")
print()
print("Dicionário salvo em reports/dicionario_abt.csv")

Reconciliação: gravado 1,851,852 | relido 1,851,852  OK

Dicionário de variáveis (22 entradas, todas descritas):
                variavel                                                                  descricao                                 derivada_de    tipo  % preenchido
                     ano                                                 ciclo da avaliação (chave)                                               Int64         100.0
            id_municipio                                           código IBGE do município (chave)                                                 str         100.0
               rede_nome                                          rede de ensino que atende o aluno                                                 str         100.0
           rede_taxa_ant    taxa de alfabetização da rede do aluno, no município, no ciclo anterior                                             float64          98.1
rede_media_portugues_ant         proficiência média em po